# 🔬 خط أنابيب ML كامل مع تتبع التجارب (Full ML Pipeline + Experiment Tracking)
### مشروع C1 — مسار علم البيانات (Data Science Track)

---
## 🎯 المشكلة التجارية (Business Problem)
فريق بيبني موديل للتنبؤ بأسعار العقارات، وبيجرّب موديلات وإعدادات كتير. المشكلة:
**"أنهي تجربة كانت الأحسن؟ وبأي إعدادات؟"** — لو مش متسجّل، النتائج بتضيع وما ينفعش تكرّرها.

**الحل:** خط أنابيب (Pipeline) منظّم + **تتبّع التجارب (Experiment Tracking) بـ MLflow** —
كل تجربة بتتسجّل: الإعدادات (params) + المقاييس (metrics) + الموديل نفسه (artifact).

## 📦 ما الذي يثبته المشروع
**Pipeline** (معالجة + موديل) · **Cross-Validation** · **MLflow** (params/metrics/model logging) ·
المقارنة واختيار الأفضل (Model Selection) · استرجاع الموديل المسجّل وتقييمه.


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "data_science/c1_mlflow_pipeline"          # مسار المشروع داخل portfolio/
PKGS    = ["mlflow", "xgboost"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| Pipeline + ColumnTransformer | Géron — *Hands-On ML* (ch.2) | يمنع تسريب المعالجة ويبسّط الكود |
| Cross-Validation | ISLR (ch.5) / Géron | تقدير أمين لأداء الموديل |
| **تتبّع التجارب (MLflow)** | Huyen — *Designing ML Systems* (ch.6) | تسجيل/مقارنة/تكرار التجارب |
| Model Registry | وثائق MLflow | حفظ الموديل كـ artifact واسترجاعه |
| مقاييس الانحدار (RMSE/MAE/R²) | ISLR (ch.3) | تقييم دقّة التنبؤ |

> 🎯 **بيُستخدم في الواقع:** أي فريق ML جاد بيستخدم تتبّع تجارب (MLflow / Weights & Biases) —
> ده جوهر **MLOps**: تكرارية، مقارنة، وحَوكمة الموديلات.


## 0️⃣ المكتبات والبيانات

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import mlflow, mlflow.sklearn
df = pd.read_csv('data/house_prices_data.csv')
print(df.shape)

## 1️⃣ استكشاف سريع + هندسة ميزة (Feature Engineering)
نضيف **عمر المنزل (house_age)** بدل سنة البناء — أكثر دلالة للموديل.

In [ ]:
# TODO: أضف house_age = 2025 - year_built، وحدّد عمود الهدف target='sale_price'
df['house_age'] = ...
target = 'sale_price'
print(df.describe())

## 2️⃣ التقسيم وخط المعالجة (Train/Test Split + Preprocessing Pipeline)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
# TODO: حدّد X و y، الأعمدة الرقمية/الفئوية، split، و ColumnTransformer
#       (انتبه: square_footage فيها قيم ناقصة → استخدم SimpleImputer داخل الـ Pipeline)
...

## 3️⃣ إعداد MLflow (تتبّع محلي)
نخلّي MLflow يسجّل في قاعدة بيانات محلية `sqlite:///mlflow.db` (مش محتاج سيرفر)، ونعمل
**تجربة (experiment)** باسم محدّد. *(ملاحظة: من MLflow 3 الـ file-store اتحوّل لوضع صيانة،
فالموصى به backend قاعدة بيانات زي sqlite.)*

In [ ]:
# TODO: عيّن tracking_uri محلي (sqlite:///mlflow.db) و set_experiment باسم
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('house-prices-regression')

## 4️⃣ تدريب ومقارنة الموديلات — كل تجربة في MLflow Run
لكل موديل: نحسب **CV-RMSE**، ندرّب، نقيّم على الاختبار (RMSE/MAE/R²)، ونسجّل **الإعدادات + المقاييس +
الموديل** في run منفصل.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
models = {'Ridge': Ridge(), 'RandomForest': RandomForestRegressor(random_state=42), 'XGBoost': XGBRegressor(random_state=42)}
# TODO: لكل موديل افتح mlflow run، احسب CV-RMSE، درّب، قيّم، وسجّل params+metrics+model
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        ...

## 5️⃣ مقارنة التجارب واختيار الأفضل (Query Runs)
نسأل MLflow عن كل الـ runs مرتّبة بـ test-RMSE — أقل قيمة = أفضل موديل.

In [ ]:
# TODO: استخدم mlflow.search_runs مرتّبة بـ test_rmse، واطبع المقارنة، وحدّد أفضل run
runs = mlflow.search_runs(order_by=['metrics.test_rmse ASC'])
best_id = runs.iloc[0]['run_id']
print(runs)

## 6️⃣ استرجاع الموديل المسجّل والتقييم (Load + Residuals)
نحمّل الموديل الأفضل من MLflow (زي ما هيحصل في الإنتاج) ونرسم البواقي (Residuals).

In [ ]:
# TODO: حمّل أفضل موديل من mlflow (runs:/{best_id}/model)، تنبّأ، وارسم البواقي
loaded = mlflow.sklearn.load_model(f'runs:/{best_id}/model')
...

## 7️⃣ الخلاصة والتوصيات (Conclusion)
- **الـ Pipeline** ضمن إن المعالجة (scaling/encoding) تتعلّم من بيانات التدريب بس — لا تسريب.
- **MLflow** سجّل كل تجربة (params + metrics + model) → نقدر نقارن ونكرّر ونرجع لأي تجربة.
- **اختيار الموديل** اتعمل بمقياس موضوعي (test-RMSE) على الـ runs المسجّلة، مش بالحدس.
- **الاسترجاع:** حمّلنا الموديل الأفضل من الـ registry — نفس آلية النشر في الإنتاج.

> 🖥️ **شوف لوحة MLflow:**
> ```bash
> mlflow ui --backend-store-uri sqlite:///mlflow.db      # ثم افتح http://localhost:5000
> ```
> ✅ **اللي اتعلمته:** Pipeline، Cross-Validation، تتبّع التجارب بـ MLflow، المقارنة، واسترجاع الموديل.
